# Judge / Extractor Agreement (Fig. 3)

Loads artifacts written by `examples/scripts/evaluation/*.py` into `results/agreement_analysis/` (string-matcher-only) and `results/agreement_analysis_llm_match/` (LLM-matched, primary) and plots them consistently.

**Structure of this notebook:**
- **Main figure (panels a/b/c)** -- the three panels reported in the manuscript.
- **Appendix** -- string-matcher-only baseline, LLM-matching methodology + validation (precision/recall), `all` vs. `loo_no_self` comparison, bootstrap sensitivity (paper- vs. material-level), self-preference bias, remaining disagreement examples, per-dimension/per-category breakdowns, and the manuscript number-check.

Run the evaluation scripts first if `results/agreement_analysis*/` are empty or stale:
```
uv run python examples/scripts/evaluation/analyze_judge_extractor_insights.py
uv run python examples/scripts/evaluation/regenerate_judge_ranking_llm_match.py
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from llm_synthesis.utils.style_utils import get_cmap, get_palette, set_style

cmap = get_cmap()
palette = get_palette()
set_style()

heat_cmap = LinearSegmentedColormap.from_list(
    "brand_sequential", [palette[6], palette[0], palette[2]]
)

RESULTS_DIR = Path("../../results/agreement_analysis")
RESULTS_DIR_LLM = Path("../../results/agreement_analysis_llm_match")
ANNOTATIONS_DIR = Path("../../annotations")
assert RESULTS_DIR.exists(), (
    f"Run the eval scripts first: {RESULTS_DIR} not found"
)
assert RESULTS_DIR_LLM.exists(), (
    f"Run regenerate_judge_ranking_llm_match.py first: {RESULTS_DIR_LLM} not found"
)

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
FIG_DIR_LLM = RESULTS_DIR_LLM / "figures"
FIG_DIR_LLM.mkdir(parents=True, exist_ok=True)

_EVAL_SCRIPTS = str(Path("../scripts/evaluation").resolve())
if _EVAL_SCRIPTS not in sys.path:
    sys.path.insert(0, _EVAL_SCRIPTS)


def load_csv(name):
    return pd.read_csv(RESULTS_DIR / name)


def load_csv_llm(name):
    return pd.read_csv(RESULTS_DIR_LLM / name)


def load_json(name):
    with open(RESULTS_DIR / name) as fh:
        return json.load(fh)


def savefig(fig, name):
    """Saves under results/agreement_analysis/figures/ as SVG + PDF."""
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")


def savefig_llm(fig, name):
    """Saves under results/agreement_analysis_llm_match/figures/ as SVG + PDF."""
    fig.savefig(FIG_DIR_LLM / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR_LLM / f"{name}.pdf", bbox_inches="tight")


_SHORT = {
    "claude-sonnet-4.6": "Claude",
    "deepseek-v3.2": "DeepSeek",
    "gemini-3-flash": "Gemini",
    "qwen3.5-397b-a17b": "Qwen",
}
_FULL = {
    "claude-sonnet-4.6": "Claude Sonnet 4.6",
    "deepseek-v3.2": "DeepSeek V3.2",
    "gemini-3-flash": "Gemini 3 Flash",
    "qwen3.5-397b-a17b": "Qwen 3.5\n397B A17B",
    "HUMAN": "Human",
}
_FULL_WRAPPED = {
    "claude-sonnet-4.6": "Claude\nSonnet 4.6",
    "deepseek-v3.2": "DeepSeek\nV3.2",
    "gemini-3-flash": "Gemini\n3 Flash",
    "qwen3.5-397b-a17b": "Qwen\n3.5 397B A17B",
}
_BORDER_STYLE = {
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 1.4,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
}

## Main figure: panels (a)/(b)/(c)

- **(a)** Mean score per rubric dimension, all 4 candidate judges + human (outlined row). Shows LLM judges cluster tightly near-ceiling while the human is more dispersed/lower -- motivates why raw judge scores alone aren't enough and agreement metrics (panel b) are needed.
- **(b)** ICC(2,1) with human, `loo_no_self` (self-scored cells excluded), **LLM-matched** materials -- the primary judge-selection metric and primary reported ranking. Single-metric bar chart (no error bars in the main figure; see Appendix for the full uncertainty analysis and why ICC(2,1) is shown alone here). Corroborating metrics (ICC(3,1), Spearman rho, Cohen kappa) agree with this ranking -- printed below the chart, full values in Appendix Table 1.
- **(c)** Concrete example: GdCo2 (paper `cond-mat.0503432`), extracted by Gemini-3-Flash, scored independently by all 5 graders (4 LLM judges + human). The extraction is judged correct by the human (synthesis method and starting materials match ground truth), yet the 5 overall_score verdicts still span 1.7 points (3.2-4.9). This pattern -- Gemini-3-Flash highest, DeepSeek-V3.2 lowest -- holds across every "correct extraction, judges still diverge" example we checked (6/6), i.e. it is systematic judge calibration, not noise from this one example.

In [ ]:
# Panel (a): judge behavior by rubric dimension.
behavior_a = load_csv("insights_judge_behavior.csv")
score_cols_a = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

heat_a = behavior_a.set_index("judge")[score_cols_a]
heat_a.columns = [
    c.replace("_score", "").replace("_", "\n") for c in heat_a.columns
]

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.heatmap(
        heat_a,
        annot=True,
        fmt=".2f",
        annot_kws={"fontsize": 9},
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "Mean score (1-5)"},
        ax=ax,
    )
    ax.set_xticklabels(
        ax.get_xticklabels(), rotation=45, ha="right", fontsize=9
    )
    ax.set_yticklabels([_FULL[j] for j in heat_a.index], fontsize=9)
    for spine_row, judge in enumerate(heat_a.index):
        if judge == "HUMAN":
            ax.add_patch(
                plt.Rectangle(
                    (0, spine_row),
                    len(heat_a.columns),
                    1,
                    fill=False,
                    edgecolor="black",
                    lw=2.5,
                )
            )
    # set y label to "Choice of Judge"
    ax.set_ylabel("Choice of Judge")
    # capitalize x tick labels, i.e. overall score -> Overall Score
    ax.set_xticklabels([c.title() for c in heat_a.columns])
    # set title
    plt.tight_layout()
    savefig(fig, "panel_a_judge_behavior_dimensions")
    plt.show()

In [ ]:
# Panel (b): ICC(2,1) with human, loo_no_self, LLM-matched (PRIMARY reported
# ranking). Single-metric bar -- one bar per judge -- rather than a 4-metric
# grouped chart: ICC(2,1) is the metric that matches the actual use case
# (does a judge's raw score substitute for a human score, not just preserve
# rank order), and it is already the manuscript's stated primary criterion.
# Corroborating metrics (rho, ICC(3,1), kappa) are printed below the chart
# and tabulated in full in the Appendix -- all four agree on this ranking's
# top 2 (Claude, then DeepSeek), which is the load-bearing claim for judge
# selection.
#
# LLM-matched: human<->LLM material names matched by string similarity first,
# then an LLM judge (DspyNameMatcherJudge) as a fallback for names the string
# matcher missed (high+medium confidence only; see Appendix for full
# methodology, validation, and how much this improves recall over
# string-matching alone).
#
# No error bars here by design: the paper-level bootstrap SE/CI genuinely is
# wide at n~19-20 papers/judge (this is a sample-size fact, not fixable by
# choice of error-bar convention -- verified against +-1 SE, 95% CI, and
# material-level bootstrap, see Appendix). Showing it in the main figure adds
# visual clutter without changing the conclusion; full uncertainty analysis
# lives in the Appendix.
from compare_multi_llm_results_llm_match import (  # noqa: E402
    build_name_matcher_judge,
    load_annotations as load_annotations_llm_match,
)
from eval_utils import merge_on_material_id  # noqa: E402

_ANNOTATIONS_DIR_STR = str(ANNOTATIONS_DIR)
_SKIP_FOLDERS = [
    "annotation_guide_catalysis",
    "2883daff26f16a13134a26ca5d366549a14fcc9c",
    "90233593a9aa72b4bacfdeadc20050ae6d4b88e1",
]

# NOTE on reproducibility: build_name_matcher_judge() calls a live LLM, so its
# medium-confidence proposals can vary slightly run-to-run (the high-confidence
# ones are stable). human_df/llm_df below are the single source of truth for
# BOTH this panel's plot AND the Appendix cells that need per-material rows
# (A2/A3) -- we do NOT also read the pre-generated CSV here, to avoid two
# numbers that can silently drift apart. If you need the exact CSV-committed
# numbers for a manuscript citation, use regenerate_judge_ranking_llm_match.py's
# output directly (results/agreement_analysis_llm_match/insights_judge_ranking_loo.csv)
# rather than re-deriving from a fresh (possibly slightly different) LLM match.
_matcher = build_name_matcher_judge("claude-sonnet-4.6")
human_df, llm_df = load_annotations_llm_match(
    _ANNOTATIONS_DIR_STR, skip_folders=_SKIP_FOLDERS, matcher=_matcher
)

from eval_utils import compute_agreement_metrics  # noqa: E402

_point_rows = []
for judge in sorted(llm_df["judge_id"].dropna().unique()):
    jdf = llm_df[(llm_df["judge_id"] == judge) & (llm_df["synth_llm"] != judge)]
    merged = merge_on_material_id(human_df, jdf, ["overall_score", "paper_id"])
    m = compute_agreement_metrics(
        merged["overall_score_h"], merged["overall_score_l"]
    )
    if not m:
        continue
    _point_rows.append(
        {
            "judge": judge,
            **{k: m[k] for k in ("icc2", "icc3", "rho", "kappa", "n")},
        }
    )

loo_llm_no_self = pd.DataFrame(_point_rows)
loo_llm_no_self["judge_short"] = loo_llm_no_self["judge"].map(_SHORT)
loo_llm_no_self["rank_icc2"] = (
    loo_llm_no_self["icc2"].rank(ascending=False, method="min").astype(int)
)
loo_llm_no_self = loo_llm_no_self.sort_values("icc2", ascending=False)

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = np.arange(len(loo_llm_no_self))
    ax.bar(x, loo_llm_no_self["icc2"], width=0.6, color=palette[0])
    for xi, (_, row) in zip(x, loo_llm_no_self.iterrows()):
        va, y = (
            ("bottom", row["icc2"] + 0.015)
            if row["icc2"] >= 0
            else ("top", row["icc2"] - 0.015)
        )
        ax.text(
            xi,
            y,
            f"#{row['rank_icc2']}\n{row['icc2']:.2f}",
            ha="center",
            va=va,
            fontsize=9,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [_FULL_WRAPPED[j] for j in loo_llm_no_self["judge"]],
        fontsize=9,
    )
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("ICC(2,1) with\ndomain expert annotations")
    ax.set_ylim(0, max(loo_llm_no_self["icc2"]) + 0.1)
    ax.margins(y=0.15)
    ax.set_xlim(-0.5, len(loo_llm_no_self) - 0.5)
    plt.tight_layout()
    savefig_llm(fig, "panel_b_icc2_only")
    plt.show()

print("ICC(2,1) ranking (primary):", " > ".join(loo_llm_no_self["judge_short"]))
print()
print("Corroborating metrics, same judge order:")
print(
    loo_llm_no_self.set_index("judge_short")[
        ["icc3", "rho", "kappa"]
    ].to_string()
)

In [ ]:
# Panel (c): concrete example -- GdCo2, extracted by Gemini-3-Flash, scored
# by all 5 graders. Correct extraction (synthesis_method, starting materials
# all match ground truth) -- judges still diverge 4.9 (Gemini) -> 3.2
# (DeepSeek), a 1.7-point spread with no extraction error to point to. The
# Gemini-highest/DeepSeek-lowest pattern holds in ALL 6 candidates checked
# (not just this one) -- see Appendix A7 for the other 5 examples, including
# the one genuine extraction error found.
import textwrap

_ann_dir = ANNOTATIONS_DIR / "cond-mat.0503432"
with open(_ann_dir / "result.json") as fh:
    _result = json.load(fh)
with open(_ann_dir / "result_human.json") as fh:
    _human = json.load(fh)

_extractor = "gemini-3-flash"
_material_name = "GdCo2"
_extractor_order = _human["extractor_order"]
_extractor_idx = _extractor_order.index(_extractor)

_human_mat = next(
    m for m in _human["materials"] if m["material_name"] == _material_name
)
_human_eval = _human_mat["evaluations"][_extractor_idx]["evaluation"]
_human_recipe = _human_mat["human_recipe"]

_llm_entry = next(e for e in _result if e["synth_llm"] == _extractor)
_mat_entry = next(
    m for m in _llm_entry["materials"] if m["material"] == _material_name
)
_extracted_synthesis = _mat_entry["synthesis"]

_verdicts = {"HUMAN": _human_eval}
for _jev in _mat_entry["evaluations"]:
    _verdicts[_jev["judge_llm"]] = _jev["evaluation"]

_source_quote = (
    _human_recipe["steps"][0]["description"]
    if _human_recipe["steps"]
    else "(no steps in human recipe)"
)
_extracted_method = _extracted_synthesis.get("synthesis_method")
_ground_truth_method = _human_recipe.get("synthesis_method")

print(
    f"=== cond-mat.0503432 | {_material_name!r} extracted by {_extractor} ==="
)
print("SOURCE (human-quoted sentence from the paper):")
print(
    textwrap.fill(
        _source_quote, width=100, initial_indent="  ", subsequent_indent="  "
    )
)
print()
print(f"EXTRACTED synthesis_method ({_extractor}):  {_extracted_method!r}")
print(
    f"GROUND TRUTH synthesis_method (human recipe):     {_ground_truth_method!r}"
)
print(
    f"  -> {'OK (matches)' if _extracted_method == _ground_truth_method else 'WRONG'}"
)
print()

_order = [
    "HUMAN",
    "gemini-3-flash",
    "claude-sonnet-4.6",
    "qwen3.5-397b-a17b",
    "deepseek-v3.2",
]
_order = [j for j in _order if j in _verdicts]
_scores = [_verdicts[j]["scores"]["overall_score"] for j in _order]
_short_names = {"HUMAN": "Human", **_SHORT}
_labels = [_short_names[j] for j in _order]
_colors = [palette[2] if j == "HUMAN" else palette[0] for j in _order]

fig, ax = plt.subplots(figsize=(5, 3.2))
_bars = ax.barh(_labels, _scores, color=_colors)
ax.bar_label(_bars, fmt="%.1f", padding=3)
ax.set_xlim(0, 5.5)
ax.set_xlabel("overall_score")
ax.set_title(f'"{_material_name}" via {_extractor}: 5 verdicts', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
savefig(fig, "panel_c_judge_disagreement_example")
plt.show()

for _j in _order:
    print(f"--- {_j} ({_verdicts[_j]['scores']['overall_score']}) ---")
    print(
        _verdicts[_j]["reasoning"]
        or _verdicts[_j]["scores"].get("overall_reasoning", "")
    )
    print()

---
# Appendix

Everything below is supplementary: robustness checks, methodology detail, and the remaining SI figures/tables. Nothing here is a main-text figure.

## A1. LLM-matching methodology + validation (precision/recall)

**Why LLM-matching exists:** the string matcher alone (SequenceMatcher + word-Jaccard, threshold 0.7, `eval_utils.find_best_matches`) links materials in only ~15-17 of the 34 annotated papers per judge -- most of the rest fail purely on phrasing, not because there is genuinely no correspondence (e.g. human `"Bi2-xSbxTe3"` vs LLM `"Bi2Te3"`/`"Sb2Te3"`, or `"HA"` vs `"Ca10(PO4)6(OH)2"` for hydroxyapatite -- same compound, different notation). This costs real statistical power: agreement-metric uncertainty is driven by how many independent *papers* are available (not materials), so recovering more papers meaningfully improves the evidence base even though the ranking itself is stable either way (A2).

**Method:** after the string matcher, an LLM judge (`DspyNameMatcherJudge` -- the same aligner `eval_vlm.py` already uses for the thermocatalysis VLM digitization eval) is run as a fallback on whatever the string matcher left unmatched. Only its **high-** and **medium-**confidence proposals are kept; low-confidence proposals were spot-checked as unreliable during development (e.g. a peptide sequence matched to an unrelated molecular formula, or the LLM's own leaked reasoning text matched as if it were a material name) and are dropped entirely.

**Validation (manual review by a domain expert against the source papers; raw annotation files shipped at `examples/scripts/evaluation/name_matcher_validation/*.csv`):**

In [ ]:
_VALIDATION_DIR = Path("../scripts/evaluation/name_matcher_validation")
high_conf = pd.read_csv(
    _VALIDATION_DIR / "high_confidence_matches_reviewed.csv"
)
med_conf = pd.read_csv(
    _VALIDATION_DIR / "medium_confidence_matches_reviewed.csv"
)
recall_check = pd.read_csv(_VALIDATION_DIR / "recall_check_reviewed.csv")

correct_col = "correct(y/n)"
high_correct = (high_conf[correct_col] == "y").sum()
med_correct = (med_conf[correct_col] == "y").sum()
recall_col = "should_have_matched(y/n)"
recall_misses = (recall_check[recall_col] == "y").sum()

validation_summary = pd.DataFrame(
    [
        {
            "check": "High-confidence match precision",
            "correct": high_correct,
            "total": len(high_conf),
            "rate": f"{high_correct / len(high_conf):.0%}",
        },
        {
            "check": "Medium-confidence match precision",
            "correct": med_correct,
            "total": len(med_conf),
            "rate": f"{med_correct / len(med_conf):.0%}",
        },
        {
            "check": "Recall spot-check (real misses / candidates sampled)",
            "correct": recall_misses,
            "total": len(recall_check),
            "rate": f"{recall_misses}/{len(recall_check)} were real misses",
        },
    ]
)
print("=== Appendix Table A1: name-matcher validation summary ===")
print(validation_summary.to_string(index=False))
print()
print("2 medium-confidence pairs manually rejected (both dropped a real")
print("structural component, e.g. 'LaAlO3/SrTiO3' -> 'LaAlO3'), denylisted")
print("by name in compare_multi_llm_results_llm_match.py _REJECTED_MATCHES:")
print(
    med_conf[med_conf[correct_col] == "n"][
        ["paper_id", "human_name", "llm_name"]
    ].to_string(index=False)
)
print()
print("NOTE (separate, unfixable-by-any-matcher limitation): some human ")
print("ground-truth names are themselves too vague to link to a specific ")
print("extraction even in principle (e.g. 'Pf-AgNPs', 'Oxidized CNS', ")
print("'Cyanine SMILES') -- the annotation guidelines did not enforce a ")
print("standardized naming/formula convention. This caps achievable n ")
print("regardless of matching method.")

## A2. Recall improvement: string-matcher-only vs. LLM-matched

How many more papers (and materials) does LLM-matching recover, and does the judge ranking change?

In [ ]:
loo_string = load_csv("insights_judge_ranking_loo.csv")
loo_string = loo_string[loo_string["cell_set"] == "loo_no_self"].set_index(
    "judge"
)
loo_llm_idx = loo_llm_no_self.set_index(
    "judge"
)  # from panel (b) -- live LLM-matched, loo_no_self

compare_table = pd.DataFrame(
    {
        "n_string_matcher": loo_string["n"],
        "n_llm_matched": loo_llm_idx["n"],
        "recall_gain": loo_llm_idx["n"] - loo_string["n"],
        "icc2_string_matcher": loo_string["icc2"],
        "icc2_llm_matched": loo_llm_idx["icc2"],
        "rho_string_matcher": loo_string["rho"],
        "rho_llm_matched": loo_llm_idx["rho"],
        "kappa_string_matcher": loo_string["kappa"],
        "kappa_llm_matched": loo_llm_idx["kappa"],
    }
).sort_values("icc2_llm_matched", ascending=False)

print(
    "=== Appendix Table A2: string-matcher-only vs. LLM-matched (loo_no_self) ==="
)
print(compare_table.to_string())
print()
print(
    "Ranking on icc2_llm_matched (primary, panel b):",
    " > ".join(compare_table.index),
)
print(
    "Ranking on icc2_string_matcher (baseline):",
    " > ".join(loo_string.sort_values("icc2", ascending=False).index),
)
print()
print(
    "-> Same top-2 (Claude, DeepSeek) either way; Qwen/Gemini swap ranks #3/#4,"
)
print(
    "   which does not affect the deployed extractor/judge pair (Qwen extractor,"
)
print("   DeepSeek judge).")

## A3. `loo_no_self` vs. `all` cells (self-preference sanity check)

`loo_no_self` excludes each judge's own self-scored cells (leave-one-out), so a judge cannot inflate its apparent quality by grading itself favorably. `all` includes every cell. Comparing the two shows whether the ranking depends on this correction -- it should, especially for Gemini-3-Flash, which shows substantial self-preference (see A6).

In [ ]:
point_rows_all = []
for judge in sorted(llm_df["judge_id"].dropna().unique()):
    jdf = llm_df[
        llm_df["judge_id"] == judge
    ]  # no synth_llm != judge filter -> self-scored cells included
    merged = merge_on_material_id(human_df, jdf, ["overall_score", "paper_id"])
    if len(merged) < 2:
        continue
    from eval_utils import compute_agreement_metrics

    m = compute_agreement_metrics(
        merged["overall_score_h"], merged["overall_score_l"]
    )
    if not m:
        continue
    point_rows_all.append(
        {
            "judge": judge,
            **{k: m[k] for k in ("icc2", "icc3", "rho", "kappa", "n")},
        }
    )

all_cells_df = (
    pd.DataFrame(point_rows_all)
    .set_index("judge")
    .sort_values("icc2", ascending=False)
)
all_cells_df.index = all_cells_df.index.map(_SHORT)

print(
    "=== Appendix Table A3: ranking with self-scored cells INCLUDED ('all') ==="
)
print(all_cells_df.to_string())
print()
print("Ranking (all, self-scored included):", " > ".join(all_cells_df.index))
print(
    "Ranking (loo_no_self, primary):     ",
    " > ".join(
        loo_llm_idx.sort_values("icc2", ascending=False).index.map(_SHORT)
    ),
)

## A4. Uncertainty analysis: paper-level bootstrap, +-1 SE vs. 95% CI vs. material-level

**Why panel (b) shows no error bars:** at n~19-20 independent papers per judge, uncertainty on a correlation-type statistic (ICC/rho/kappa) is genuinely wide -- this is a direct consequence of sample size, not an artifact of which error-bar convention is used. Below: the paper-level bootstrap (correct unit -- see next paragraph), shown as +-1 SE and as a 95% CI, plus a material-level bootstrap shown only as a labeled sensitivity check (NOT a valid alternative).

**Why the bootstrap resamples whole papers, not individual materials:** materials within the same paper share extraction context, chemistry difficulty, and the same annotator's reading of that paper -- they are not independent observations. Resampling individual materials directly (ignoring paper boundaries) implicitly assumes any material could have come from any paper, which is false, and produces a tighter but overconfident error bar purely by over-counting correlated observations as independent (pseudo-replication). A toy simulation (paper-to-paper true differences only 3x the within-paper material noise -- a conservative assumption, since real papers plausibly differ far more) showed material-level bootstrap SE understating the true paper-level SE by ~20% under those conditions.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score
from eval_utils import categorize_score

_N_BOOT = 2000
_RNG = np.random.default_rng(0)


def _icc2_icc3(h, l):
    """Closed-form 2-rater ANOVA ICC (exact match to pingouin for n=2 raters,
    ~150x cheaper per call -- needed since the bootstrap runs thousands of calls)."""
    if len(h) < 5:
        return float("nan"), float("nan")
    n, k = len(h), 2
    data = np.column_stack([h, l])
    grand_mean, subj_means, rater_means = (
        data.mean(),
        data.mean(axis=1),
        data.mean(axis=0),
    )
    ss_subj = k * ((subj_means - grand_mean) ** 2).sum()
    ss_rater = n * ((rater_means - grand_mean) ** 2).sum()
    ss_error = ((data - grand_mean) ** 2).sum() - ss_subj - ss_rater
    ms_subj, ms_rater, ms_error = (
        ss_subj / (n - 1),
        ss_rater / (k - 1),
        ss_error / ((n - 1) * (k - 1)),
    )
    icc2 = (ms_subj - ms_error) / (
        ms_subj + (k - 1) * ms_error + k * (ms_rater - ms_error) / n
    )
    icc3 = (ms_subj - ms_error) / (ms_subj + (k - 1) * ms_error)
    return icc2, icc3


def _agreement_metrics(h, l):
    paired = pd.DataFrame({"h": h, "l": l}).dropna()
    if len(paired) < 2:
        return None
    h_arr, l_arr = paired["h"].to_numpy(float), paired["l"].to_numpy(float)
    rho = (
        spearmanr(h_arr, l_arr).statistic
        if np.unique(h_arr).size > 1
        else np.nan
    )
    kappa = cohen_kappa_score(
        paired["h"].apply(categorize_score),
        paired["l"].apply(categorize_score),
        weights="quadratic",
    )
    icc2, icc3 = _icc2_icc3(h_arr, l_arr)
    return {
        "rho": rho,
        "kappa": kappa,
        "icc2": icc2,
        "icc3": icc3,
        "n": len(paired),
    }


point_rows, boot_rows_paper, boot_rows_material = [], [], []
for judge in sorted(llm_df["judge_id"].dropna().unique()):
    jdf = llm_df[(llm_df["judge_id"] == judge) & (llm_df["synth_llm"] != judge)]
    merged = merge_on_material_id(human_df, jdf, ["overall_score", "paper_id"])
    m = _agreement_metrics(merged["overall_score_h"], merged["overall_score_l"])
    if not m:
        continue
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    paper_groups = {p: g for p, g in merged.groupby("paper_id_h")}
    paper_ids = list(paper_groups)
    n_rows = len(merged)
    for _ in range(_N_BOOT):
        draw_p = _RNG.choice(paper_ids, size=len(paper_ids), replace=True)
        resampled_p = pd.concat(
            [paper_groups[p] for p in draw_p], ignore_index=True
        )
        bm_p = _agreement_metrics(
            resampled_p["overall_score_h"], resampled_p["overall_score_l"]
        )
        draw_m = _RNG.choice(n_rows, size=n_rows, replace=True)
        resampled_m = merged.iloc[draw_m]
        bm_m = _agreement_metrics(
            resampled_m["overall_score_h"], resampled_m["overall_score_l"]
        )
        for metric in ("icc2", "icc3", "rho", "kappa"):
            if bm_p:
                boot_rows_paper.append(
                    {"judge": judge, "metric": metric, "value": bm_p[metric]}
                )
            if bm_m:
                boot_rows_material.append(
                    {"judge": judge, "metric": metric, "value": bm_m[metric]}
                )

point_df = pd.DataFrame(point_rows)
boot_df_paper = pd.DataFrame(boot_rows_paper).dropna()
boot_df_material = pd.DataFrame(boot_rows_material).dropna()

se_paper = (
    boot_df_paper.groupby(["judge", "metric"])["value"].std().rename("se_paper")
)
se_material = (
    boot_df_material.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se_material")
)
ci_paper = (
    boot_df_paper.groupby(["judge", "metric"])["value"]
    .quantile([0.025, 0.975])
    .unstack()
)
ci_paper.columns = ["ci95_lo", "ci95_hi"]

se_compare = pd.concat([se_paper, se_material, ci_paper], axis=1).reset_index()
se_compare["ci95_halfwidth"] = (
    se_compare["ci95_hi"] - se_compare["ci95_lo"]
) / 2
se_compare["se_ratio_material_over_paper"] = (
    se_compare["se_material"] / se_compare["se_paper"]
)
se_compare["judge_short"] = se_compare["judge"].map(_SHORT)
se_icc2 = se_compare[se_compare["metric"] == "icc2"].sort_values(
    "se_paper", ascending=False
)

print(
    "=== Appendix Table A4: paper-level vs. material-level bootstrap uncertainty (ICC2, loo_no_self) ==="
)
print(
    "Material-level SE is expected to be SMALLER -- that IS the overconfidence being"
)
print("flagged, not evidence the paper-level numbers are wrong.\n")
print(
    se_icc2[
        [
            "judge_short",
            "se_paper",
            "ci95_halfwidth",
            "se_material",
            "se_ratio_material_over_paper",
        ]
    ]
    .rename(columns={"judge_short": "judge"})
    .to_string(index=False)
)

## A5. Extractor choice robustness: ranking agreement across graders

Spearman correlation between each judge's (including HUMAN's) extractor ranking and the human ranking. High values across the board mean the extractor choice (Qwen3.5 as the deployed extractor) is not an artifact of which judge you trust.

In [ ]:
ranking_by_judge = load_csv("insights_extractor_ranking_by_judge.csv")

fig, ax = plt.subplots(figsize=(7, 4))
order = ranking_by_judge.sort_values("spearman_vs_human", ascending=False)[
    "grader"
]
sns.barplot(
    data=ranking_by_judge,
    x="grader",
    y="spearman_vs_human",
    order=order,
    ax=ax,
    color=palette[2],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Spearman rho vs. human extractor ranking")
ax.set_title("Extractor ranking agreement with human, per grader")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig(fig, "appendix_extractor_ranking_agreement")
plt.show()

quality = load_csv("insights_extractor_quality.csv").set_index("extractor")
print(
    quality[["human_overall"]]
    .sort_values("human_overall", ascending=False)
    .to_string()
)

## A6. Self-preference bias

Does a judge score its own extractions higher than it scores others'?

In [ ]:
self_pref = load_csv("insights_self_preference.csv")
self_bias = load_csv("insights_self_bias_did.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_df_pref = self_pref.melt(
    id_vars="model",
    value_vars=["self_score_mean", "peer_score_mean"],
    var_name="target",
    value_name="score",
)
sns.barplot(
    data=plot_df_pref,
    x="model",
    y="score",
    hue="target",
    ax=axes[0],
    palette=palette[:2],
)
axes[0].set_title("Self vs. peer scoring")
axes[0].set_ylabel("mean overall_score")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=self_bias, x="model", y="self_bias_did", ax=axes[1], color=palette[0]
)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Self-bias (DiD vs. human)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig(fig, "appendix_self_preference_bias")
plt.show()

for _, row in self_pref.iterrows():
    did = self_bias.set_index("model").loc[row["model"], "self_bias_did"]
    print(
        f"{row['model']:20s} self={row['self_score_mean']:.3f}  peer={row['peer_score_mean']:.3f}  "
        f"raw_diff={row['self_preference']:+.3f}  DiD_vs_human={did:+.3f}"
    )

## A7. Remaining judge-disagreement examples

The genuine-error example (a real, checkable extraction mistake, kept separate from the "correct extraction, judges still diverge" pattern) plus the other 5 "correct extraction, judges still diverge" candidates checked (panel c used the strongest of these 6). All 6 show the same Gemini-highest/DeepSeek-lowest calibration pattern.

### A7 summary: shortened judge quotes, all 6 candidates

Full reasoning traces are printed by each `show_judge_disagreement_example(...)` call below. Shortened quotes for quick reference:

**1. Bi1.74Pb0.38Sr1.88CuO6+δ** via Gemini-3-Flash — **genuine error** (`'flux growth'` vs actual `'floating-zone technique'`)
- Human (SC 4.5): "Matched... extracted spurious second material Bi2Sr2CuO6+δ."
- Gemini, permissive (SC 4.9): "...accurately identifies the core synthesis method (**floating-zone technique**)..." — contradicts its own extracted field.
- DeepSeek, harsh (SC 2.5): "...hallucination... classifying the method as **'flux growth'**... absence is not an error has been violated."

**2. WFe2Ni-red** via DeepSeek-V3.2 — correct extraction
- Human (SC 4.75): "Wrong compound type."
- Gemini, permissive (SC 5.0): "...leaves fields null where information is absent, adhering to the core evaluation principle."
- DeepSeek, harsh, self-scoring (SC 2.9): "...incomplete, missing critical quantitative details... implied by standard synthetic practice, even if not explicitly stated."

**3. (La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3** via Gemini-3-Flash — correct extraction
- Human (SC 4.0): "...type 'two-dimensional materials' imprecise, duration inferred, step 1 conditions absent."
- Gemini, permissive, self-scoring (SC 4.9): "...successfully identified the target material, synthesis method (PLD)... captured the complex growth conditions."
- DeepSeek, harsh (SC 2.9): "...significant structural issues: incorrectly populates the 'starting_materials' list... conflates specific conditions."

**4. SrTiO3** via Gemini-3-Flash — correct extraction
- Human (SC 4.75): (no written comment)
- Gemini, permissive, self-scoring (SC 4.9): "...correctly differentiated between the treatment of Sample A and Sample B... highly accurate."
- DeepSeek, harsh (SC 3.2): "...incorrectly lists 'buffered oxide etch' as a starting material... over-extraction beyond the synthesis context."

**5. GdCo2** via Gemini-3-Flash — correct extraction
- Human (SC 4.5): "Clean extraction with 2 steps, correct materials. The synthesis_method contains 'induction melting'."
- Gemini, permissive, self-scoring (SC 4.9): "...correctly identified the starting materials... the synthesis section is brief but well-captured."
- DeepSeek, harsh (SC 3.2): "...incorrectly classifies the synthesis method as 'arc melting & induction melting' when the text only states 'melting'... hallucinates a 'mix' action."

**6. 5-AGNR** via Gemini-3-Flash — correct extraction
- Human (SC 4.75): (no written comment)
- Gemini, permissive, self-scoring (SC 5.0): "Performed exceptionally well... correctly identified the specific molecular precursor (DBP) and the distinct temperature (350°C)."
- DeepSeek, harsh (SC 3.4): "...misses the crucial detail that the synthesis occurs 'on a metal surface'... fails to capture the sequential nature of the process."

**Pattern across all 6:** Gemini-3-Flash is consistently the highest, near-ceiling scorer (4.9–5.0 every time), and its reasoning reads as generic praise for what is *present* rather than flagging what is wrong or missing — in example 1 this leads it to describe the extraction as if it captured the correct method, when the extracted field itself is wrong. DeepSeek-V3.2 is consistently the harshest, and the only judge whose reasoning explicitly hunts for hallucinations/omissions, including catching the one genuine error. This is a systematic difference in judge calibration (generous vs. adversarial reading), not example-specific noise — the direct motivation for the judge-selection result in panel (b).


In [ ]:
import textwrap as _textwrap

_SHORT_NAME = {"HUMAN": "Human", **_SHORT}


def show_judge_disagreement_example(
    paper_id, material_name, extractor, save_name=None
):
    """Print source quote + extracted/ground-truth method + bar chart of all verdicts."""
    ann_dir = ANNOTATIONS_DIR / paper_id
    with open(ann_dir / "result.json") as fh:
        result = json.load(fh)
    with open(ann_dir / "result_human.json") as fh:
        human = json.load(fh)

    extractor_order = human["extractor_order"]
    extractor_idx = extractor_order.index(extractor)

    human_mat = next(
        m for m in human["materials"] if m["material_name"] == material_name
    )
    human_eval = human_mat["evaluations"][extractor_idx]["evaluation"]
    human_recipe = human_mat["human_recipe"]

    llm_entry = next(e for e in result if e["synth_llm"] == extractor)
    mat_entry = next(
        m for m in llm_entry["materials"] if m["material"] == material_name
    )
    extracted_synthesis = mat_entry["synthesis"]

    verdicts = {"HUMAN": human_eval}
    for jev in mat_entry["evaluations"]:
        verdicts[jev["judge_llm"]] = jev["evaluation"]

    source_quote = (
        human_recipe["steps"][0]["description"]
        if human_recipe["steps"]
        else "(no steps in human recipe)"
    )
    extracted_method = extracted_synthesis.get("synthesis_method")
    ground_truth_method = human_recipe.get("synthesis_method")

    print(f"=== {paper_id} | {material_name!r} extracted by {extractor} ===")
    print("SOURCE (human-quoted sentence from the paper):")
    print(
        _textwrap.fill(
            source_quote, width=100, initial_indent="  ", subsequent_indent="  "
        )
    )
    print()
    print(f"EXTRACTED synthesis_method ({extractor}):  {extracted_method!r}")
    print(
        f"GROUND TRUTH synthesis_method (human recipe):     {ground_truth_method!r}"
    )
    match = (
        "OK (matches)" if extracted_method == ground_truth_method else "WRONG"
    )
    print(f"  -> {match}")
    print()

    order = [
        j
        for j in [
            "HUMAN",
            "gemini-3-flash",
            "claude-sonnet-4.6",
            "qwen3.5-397b-a17b",
            "deepseek-v3.2",
        ]
        if j in verdicts
    ]
    scores = [verdicts[j]["scores"]["overall_score"] for j in order]
    labels = [_SHORT_NAME[j] for j in order]
    colors = [palette[2] if j == "HUMAN" else palette[0] for j in order]

    fig, ax = plt.subplots(figsize=(5, 3.2))
    bars = ax.barh(labels, scores, color=colors)
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_xlim(0, 5.5)
    ax.set_xlabel("overall_score")
    short_material = (
        material_name
        if len(material_name) <= 25
        else material_name[:22] + "..."
    )
    ax.set_title(f'"{short_material}" via {extractor}: 5 verdicts', fontsize=10)
    ax.invert_yaxis()
    plt.tight_layout()
    if save_name:
        savefig(fig, save_name)
    plt.show()

    for j in order:
        print(f"--- {j} ({verdicts[j]['scores']['overall_score']}) ---")
        print(
            verdicts[j]["reasoning"]
            or verdicts[j]["scores"].get("overall_reasoning", "")
        )
        print()

    return verdicts


# Genuine-error example (Bi-Pb-Sr-Cu-O): extraction says \'flux growth\',
# ground truth is \'float zone & Bridgman\' -- a real, checkable error, unlike
# the other 5 examples below where the extraction is correct.
show_judge_disagreement_example(
    "cond-mat.0602418",
    "Bi1.74Pb0.38Sr1.88CuO6+\u03b4",
    "gemini-3-flash",
    save_name="appendix_genuine_error_example",
)

In [ ]:
show_judge_disagreement_example(
    "1605.04038",
    "(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3",
    "gemini-3-flash",
    save_name="appendix_candidate_example_2",
)

In [ ]:
show_judge_disagreement_example(
    "1706.00484",
    "SrTiO3",
    "gemini-3-flash",
    save_name="appendix_candidate_example_3",
)

In [ ]:
show_judge_disagreement_example(
    "cond-mat.0503432",
    "GdCo2",
    "gemini-3-flash",
    save_name="appendix_candidate_example_4",
)

In [ ]:
show_judge_disagreement_example(
    "1902.03049",
    "5-AGNR",
    "gemini-3-flash",
    save_name="appendix_candidate_example_5",
)

In [ ]:
show_judge_disagreement_example(
    "64b40972b605c6803bd37ab4",
    "WFe2Ni-red",
    "deepseek-v3.2",
    save_name="appendix_candidate_example_6",
)

## A8. Extractor x Judge score matrix, inter-judge agreement, and per-dimension/per-category breakdowns

Original SI content, unchanged from the exploratory analysis.

In [ ]:
matrix = load_csv("insights_judge_extractor_matrix.csv").set_index("synth_llm")
matrix.index.name = "extractor"
matrix.columns.name = "judge"

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
ax.set_title("Extractor x Judge overall_score (diagonal = self-scoring)")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge")
plt.show()

In [ ]:
import numpy as np

spearman = load_csv("insights_interjudge_spearman.csv").set_index("Unnamed: 0")
spearman.index.name = None
mask = np.triu(np.ones_like(spearman, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    spearman,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    mask=mask,
    cbar_kws={"label": "Spearman rho"},
    ax=ax,
)
ax.set_title("Inter-judge rank correlation")
plt.tight_layout()
savefig(fig, "heatmap_interjudge_spearman")
plt.show()

In [ ]:
quality = load_csv("insights_extractor_quality.csv").set_index("extractor")

# human_overall is indexed by extractor (the human's score of that extractor's
# output), not by judge -- it belongs as an extra column, not a judge-row.
matrix_with_human = matrix.copy()
matrix_with_human["HUMAN"] = quality["human_overall"].reindex(matrix.index)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(
    matrix_with_human,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
# outline diagonal self-scoring cells
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
# outline the human reference column
ax.add_patch(
    plt.Rectangle(
        (len(matrix_with_human.columns) - 1, 0),
        1,
        len(matrix_with_human),
        fill=False,
        edgecolor=palette[2],
        lw=2.5,
    )
)
ax.set_title("Extractor performance across judges, with human reference")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge_with_human")
plt.show()

In [ ]:
dims = load_csv("insights_dimension_means.csv").melt(
    id_vars="dimension", var_name="source", value_name="mean_score"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=dims,
    y="dimension",
    x="mean_score",
    hue="source",
    ax=ax,
    palette=palette[:2],
)
ax.set_xlim(0, 5)
ax.set_title("Judges (pooled) vs. human, by rubric dimension")
plt.tight_layout()
savefig(fig, "bar_dimension_means")
plt.show()

In [ ]:
dimension_score_cols = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

behavior_dims = load_csv("insights_judge_behavior.csv").set_index("judge")[
    dimension_score_cols
]
behavior_dims.columns = [
    c.replace("_score", "").replace("_", " ") for c in behavior_dims.columns
]

human_row = behavior_dims.loc["HUMAN"]
judge_dims = behavior_dims.drop(index="HUMAN")
abs_diff_dims = (judge_dims - human_row).abs()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    abs_diff_dims,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "abs_diff vs. human"},
    ax=ax,
)
ax.set_title("|judge - human| per rubric dimension")
plt.tight_layout()
savefig(fig, "heatmap_dimension_abs_diff")
plt.show()

In [ ]:
by_category = load_csv("multi_llm_agreement_by_material_category.csv")

for category_type, group in by_category.groupby("category_type"):
    n_per_category = group.groupby("category")["n"].first()
    print(f"{category_type} -- n materials per category:")
    print(n_per_category.to_string())
    print()

    # abs_diff heatmap (judge vs. human distance)
    pivot = group.pivot(index="category", columns="judge", values="abs_diff")
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(pivot) + 2))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "abs_diff vs. human"},
        ax=ax,
    )
    ax.set_title(f"|judge - human| by {category_type}")
    plt.tight_layout()
    slug = category_type.lower().replace(" ", "_")
    savefig(fig, f"heatmap_agreement_by_{slug}")
    plt.show()

    # raw mean scores, with HUMAN reference column (h_mean is constant across
    # judges within a category, so any row's value works)
    means_pivot = group.pivot(
        index="category", columns="judge", values="l_mean"
    )
    means_pivot["HUMAN"] = group.groupby("category")["h_mean"].first()

    fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(means_pivot) + 2))
    sns.heatmap(
        means_pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "mean overall_score"},
        ax=ax,
    )
    ax.add_patch(
        plt.Rectangle(
            (len(means_pivot.columns) - 1, 0),
            1,
            len(means_pivot),
            fill=False,
            edgecolor=palette[2],
            lw=2.5,
        )
    )
    ax.set_title(f"Mean score by {category_type}, with human reference")
    plt.tight_layout()
    savefig(fig, f"heatmap_mean_score_by_{slug}_with_human")
    plt.show()

## A9. Manuscript number check: every value cited in main.tex / supp.tex, printed from source

Single source-of-truth cell -- if a number here doesn't match the manuscript text, the manuscript is stale, not this cell.

In [ ]:
print(
    "=== Judge selection (main.tex Sec. LLM model selection; supp.tex Table human-llm-comparison) ==="
)
loo_no_self = load_csv("insights_judge_ranking_loo.csv")
loo_no_self = loo_no_self[loo_no_self["cell_set"] == "loo_no_self"].sort_values(
    "icc2", ascending=False
)
print(
    loo_no_self[
        ["judge", "n", "icc2", "icc3", "rho", "kappa", "mean_diff"]
    ].to_string(index=False)
)
best_judge = loo_no_self.iloc[0]
print(
    f"\n-> Selected judge: {best_judge['judge']} (ICC2={best_judge['icc2']:.3f}, rho={best_judge['rho']:.3f}), n={int(best_judge['n'])}"
)

print("\n=== Self-preference bias (main.tex + supp.tex) ===")
self_pref = load_csv("insights_self_preference.csv")
self_bias = load_csv("insights_self_bias_did.csv").set_index("model")
for _, row in self_pref.iterrows():
    did = self_bias.loc[row["model"], "self_bias_did"]
    print(
        f"{row['model']:20s} self={row['self_score_mean']:.3f}  peer={row['peer_score_mean']:.3f}  "
        f"raw_diff={row['self_preference']:+.3f}  DiD_vs_human={did:+.3f}"
    )

print(
    "\n=== Extractor selection: top extractor by human, ranking reproduced by every judge (main.tex) ==="
)
quality = load_csv("insights_extractor_quality.csv").set_index("extractor")
print(
    quality[["human_overall"]]
    .sort_values("human_overall", ascending=False)
    .to_string()
)
ranking_by_judge = load_csv("insights_extractor_ranking_by_judge.csv")
print()
print(
    ranking_by_judge[
        ["grader", "top_extractor", "spearman_vs_human"]
    ].to_string(index=False)
)

print("\n=== 4x4 extractor-judge matrix size, n held-out procedures ===")
print(
    f"Matrix shape: {matrix.shape[0]}x{matrix.shape[1]} (extractors x judges)"
)
print(
    f"n = {int(loo_no_self['n'].max())} max (varies per judge -- self-scored cells excluded per row in loo_no_self)"
)
print(
    "n = 59 used for the pooled 'all' cell_set (all judges, cross-tabulated over the same held-out set)"
)